# **Cascading Deletion in Hierarchical GraphRAG Memory Systems**

This notebook is the final reproducible implementation for the project **Cascading Deletion in Hierarchical GraphRAG Memory Systems**. The goal is to study whether a long-term conversational memory system can forget a target episode without rebuilding the entire memory graph, while still preserving useful non-deleted memory.

The project investigates **episode-level right-to-forget behavior** in a hierarchical GraphRAG-style memory system. A LongMemEval conversation is converted into a typed memory graph containing episodes, spans, entities, claims, communities, and summaries. When an episode is deleted, the notebook compares three deletion strategies:

1. **Naive deletion**: mark only the target episode as deleted.
2. **Cascade deletion with local restructuring**: delete the episode, invalidate supported claims, mark affected communities dirty, locally restructure the graph, and regenerate affected summaries.
3. **Full rebuild**: rebuild the graph and summaries from the remaining active episodes after removing the forgotten episode.

The final project evaluation used:

- Dataset: `xiaowu0162/longmemeval-cleaned`
- Split/file: `longmemeval_s_cleaned.json`
- Number of instances: **25**
- Maximum sessions per instance: **100**
- Maximum extraction episodes: **50**
- Number of episode deletion trials: **50**
- Number of evaluation queries per episode deletion: **5**
- Model: **Grok 4.3**
- No model training; all operations are based on structured generation, graph construction, retrieval, deletion, and evaluation.

The final results showed that cascade deletion achieved the best practical trade-off: it reduced leakage compared with naive deletion and approached full-rebuild quality while regenerating far fewer summaries.

Final aggregate scores:

| Method | Deleted leak rate | Mixed leak rate | Preserved utility | Deleted abstention | Avg. latency | Avg. summaries regenerated |
|---|---:|---:|---:|---:|---:|---:|
| Naive | 0.22 | 0.14 | 0.75 | 0.80 | 0.045s | 0.0 |
| Cascade local restructure | 0.14 | 0.06 | 0.86 | 0.90 | 183.809s | 11.72 |
| Full rebuild | 0.12 | 0.07 | 0.84 | 0.96 | 431.063s | 127.4 |

To reproduce the final submission run, use the Hugging Face dataset source, set `num_instances = 25`, `max_sessions_per_instance = 100`, and `max_extraction_episodes = 50` in the run configuration cell. The notebook saves evaluation outputs under `outputs/mvp_run/`.

In [ ]:
# Install dependencies with pip only.
# - GPU FAISS package for CUDA 12 is faiss-gpu-cu12.
# - If faiss-gpu-cu12 is unavailable on your platform, fall back to faiss-cpu.

%pip install -U \
  xai-sdk \
  openai \
  huggingface_hub \
  datasets \
  pandas \
  numpy \
  tqdm \
  pydantic \
  tenacity \
  networkx \
  python-louvain \
  scikit-learn \
  sentence-transformers \
  transformers \
  accelerate \
  torch \
  faiss-gpu-cu12

# Optional fallback if the FAISS GPU wheel fails on your environment:
# %pip install -U faiss-cpu

In [ ]:
from __future__ import annotations

import asyncio
import copy
import dataclasses
import hashlib
import json
import math
import os
import random
import re
import sqlite3
import textwrap
import time
from collections import defaultdict, deque
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple, Union

import nest_asyncio
nest_asyncio.apply()

import numpy as np
import pandas as pd
import networkx as nx
from tqdm.auto import tqdm

from pydantic import BaseModel, Field, ConfigDict, ValidationError

try:
    import torch
except Exception:
    torch = None

try:
    from sentence_transformers import SentenceTransformer
except Exception:
    SentenceTransformer = None

try:
    import faiss
except Exception:
    faiss = None

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception:
    TfidfVectorizer = None
    cosine_similarity = None

try:
    from openai import AsyncOpenAI
except Exception:
    AsyncOpenAI = None

try:
    from huggingface_hub import hf_hub_download
except Exception:
    hf_hub_download = None

try:
    from datasets import load_dataset
except Exception:
    load_dataset = None


In [ ]:
@dataclass
class MVPConfig:
    # Dataset source.
    # - "hf": download from Hugging Face repo xiaowu0162/longmemeval-cleaned.
    # - "local": read cfg.longmemeval_json_path.
    # - "synthetic": use synthetic smoke-test rows.
    dataset_source: str = "hf"

    # Hugging Face LongMemEval-cleaned settings.
    hf_dataset_id: str = "xiaowu0162/longmemeval-cleaned"
    hf_filename: str = "longmemeval_s_cleaned.json"     # smaller cleaned split; use longmemeval_m_cleaned.json for larger runs.
    hf_split: str = "longmemeval_s_cleaned"             # used only when trying datasets.load_dataset.
    hf_revision: str = "main"
    hf_cache_dir: str = "data/hf_cache"
    hf_token_env: str = "HF_TOKEN"                      # optional; public repo should not require it.
    hf_try_datasets_loader_first: bool = False           # direct JSON download is more robust for this repo.

    # Optional local fallback / override.
    longmemeval_json_path: str = "data/raw/longmemeval_s_cleaned.json"

    # Paths
    output_dir: str = "outputs/mvp_run"
    cache_path: str = "outputs/mvp_run/grok_cache.sqlite"

    # xAI / Grok
    xai_api_key_env: str = "XAI_API_KEY"
    xai_base_url: str = "https://api.x.ai/v1"
    grok_model: str = "grok-4.3"
    temperature: float = 0.0
    max_retries: int = 5
    rpm_limit: int = 1000
    max_concurrency: int = 64
    request_timeout_s: float = 90.0

    # Embeddings
    embedding_model_name: str = "BAAI/bge-large-en-v1.5"
    embedding_batch_size: int = 64
    normalize_embeddings: bool = True
    use_gpu_for_embeddings: bool = True

    # Experiment sizing
    num_instances: int = 10          # Increase for actual experiments: 50, 100, 300.
    max_sessions_per_instance: int = 12  # LongMemEval-S may have ~40 sessions; keep MVP cheap.
    max_turns_per_session: int = 20
    max_extraction_episodes: int = 250   # Safety cap per run; raise for full experiment.

    # Graph / community
    louvain_resolution: float = 1.0
    louvain_seed: int = 13
    local_restructure_hops: int = 1
    min_local_recluster_nodes: int = 3

    # Retrieval / answering
    top_k_dense: int = 12
    top_k_tfidf: int = 24
    final_context_k: int = 12
    max_context_chars: int = 14000

    # Evaluation
    deleted_direct_queries_per_target: int = 2
    deleted_paraphrase_queries_per_target: int = 3
    mixed_queries_per_target: int = 2
    run_semantic_judge: bool = True
    run_utility_judge: bool = True

    # Development controls
    use_mock_llm: bool = False       # Set True for smoke tests without API calls.
    random_seed: int = 13

cfg = MVPConfig()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
random.seed(cfg.random_seed)
np.random.seed(cfg.random_seed)
print(cfg)


## 1. Configuration, data model, and schema definitions

This section defines the experiment configuration and the internal data structures used throughout the notebook. The `MVPConfig` object controls the dataset source, Hugging Face file name, Grok API settings, embedding configuration, graph hyperparameters, retrieval settings, and evaluation settings.

The project uses a provenance-heavy memory model. This is essential because forgetting is not just raw text deletion: if a deleted episode produced claims, communities, or summaries, those derived artifacts must also be updated or invalidated.

The main objects are:

- **Episode**: one dialogue turn or conversational event used as an atomic memory unit.
- **SpanRecord**: an exact text span extracted from an episode, used for provenance.
- **EntityRecord**: a canonical entity such as the user, a person, a place, an object, or an organization.
- **ClaimRecord**: an atomic memory fact supported by one or more spans and episodes.
- **CommunityRecord**: a cluster of related claims and entities.
- **SummaryRecord**: a generated local or global summary derived from claims, entities, or child summaries.
- **ForgetSet**: the deletion target and its evaluation queries.
- **MemoryState**: the full graph memory for one LongMemEval instance.

The key design decision is that every claim stores its supporting spans and episodes, and every summary stores its input claims, child summaries, and source spans. This provenance enables the cascade deletion algorithm later in the notebook.

In [ ]:
def stable_id(prefix: str, *parts: Any, n: int = 12) -> str:
    raw = "||".join(str(p) for p in parts)
    return f"{prefix}_{hashlib.sha1(raw.encode('utf-8')).hexdigest()[:n]}"


def norm_text(s: Optional[str]) -> str:
    if not s:
        return ""
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9@._+\-:/ ]", "", s)
    return s


def safe_json_dumps(x: Any) -> str:
    return json.dumps(x, ensure_ascii=False, sort_keys=True, default=str)


def now_ms() -> int:
    return int(time.time() * 1000)


def truncate(s: str, n: int = 1000) -> str:
    if len(s) <= n:
        return s
    return s[: n - 20] + " ...[truncated]..."


def atomic_write_json(path: Union[str, Path], obj: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    tmp.replace(path)


def read_json_or_jsonl(path: Union[str, Path]) -> Any:
    path = Path(path)
    text = path.read_text(encoding="utf-8")
    stripped = text.strip()
    if stripped.startswith("["):
        return json.loads(stripped)
    rows = []
    for line in stripped.splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def validate_or_find_span(text: str, span_text: str, char_start: Optional[int] = None, char_end: Optional[int] = None) -> Tuple[int, int]:
    """Validate model-provided offsets; fall back to substring search."""
    if span_text is None:
        span_text = ""
    span_text = span_text.strip()
    if not span_text:
        return -1, -1
    if isinstance(char_start, int) and isinstance(char_end, int):
        if 0 <= char_start < char_end <= len(text) and text[char_start:char_end].strip() == span_text:
            return char_start, char_end
    idx = text.find(span_text)
    if idx >= 0:
        return idx, idx + len(span_text)
    # Case-insensitive fallback.
    idx = text.lower().find(span_text.lower())
    if idx >= 0:
        return idx, idx + len(span_text)
    return -1, -1

In [ ]:
# ---------- Pydantic models returned by Grok structured generation ----------

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class ExtractedSpan(StrictModel):
    text: str = Field(description="Exact substring from the episode text, if possible")
    char_start: int = Field(description="Character start offset in the episode text, or -1 if unknown")
    char_end: int = Field(description="Character end offset in the episode text, or -1 if unknown")
    span_type: str = Field(description="Type such as person, location, organization, preference, date, phone, address, event, other")
    normalized_text: str = Field(description="Lowercase normalized representation")
    sensitivity: str = Field(description="none, low, personal, pii, sensitive")
    forget_candidate: bool = Field(description="Whether this span is plausible as a forget target")


class ExtractedEntity(StrictModel):
    temp_id: str
    name: str
    entity_type: str
    aliases: List[str]
    source_span_texts: List[str]


class ExtractedClaim(StrictModel):
    temp_id: str
    subject: str
    predicate: str
    object: str
    claim_text: str
    supporting_span_texts: List[str]
    sensitivity: str
    forget_candidate: bool
    valid_from: str
    valid_to: str


class EpisodeExtraction(StrictModel):
    spans: List[ExtractedSpan]
    entities: List[ExtractedEntity]
    claims: List[ExtractedClaim]


class CommunitySummaryOutput(StrictModel):
    summary: str
    salient_claim_ids: List[str]
    omitted_or_uncertain: List[str]


class EpisodeForgetQuerySet(StrictModel):
    direct_deleted_questions: List[str]
    paraphrased_deleted_questions: List[str]
    mixed_questions: List[str]
    preserved_fact_questions: List[str]
    forbidden_aliases: List[str]


class AnswerOutput(StrictModel):
    answer: str
    used_memory_ids: List[str]
    abstained: bool
    confidence: float


class LeakageJudgeOutput(StrictModel):
    reveals_deleted_information: bool
    severity: str = Field(description="none, partial, full, hallucinated_specific")
    rationale: str


class UtilityJudgeOutput(StrictModel):
    correct: bool
    partially_correct: bool
    abstained_when_should_answer: bool
    rationale: str

In [ ]:
# ---------- Internal memory records ----------

@dataclass
class Episode:
    episode_id: str
    instance_id: str
    session_id: str
    turn_id: int
    role: str
    content: str
    date: str = ""
    status: str = "active"  # active | deleted
    redacted_content: Optional[str] = None
    has_answer: bool = False

    @property
    def active_text(self) -> str:
        if self.status != "active":
            return ""
        return self.redacted_content if self.redacted_content is not None else self.content


@dataclass
class SpanRecord:
    span_id: str
    episode_id: str
    text: str
    char_start: int
    char_end: int
    span_type: str
    normalized_text: str
    sensitivity: str
    forget_candidate: bool
    status: str = "active"


@dataclass
class EntityRecord:
    entity_id: str
    canonical_name: str
    entity_type: str
    aliases: List[str] = field(default_factory=list)
    source_span_ids: List[str] = field(default_factory=list)
    community_ids: List[str] = field(default_factory=list)
    status: str = "active"


@dataclass
class ClaimRecord:
    claim_id: str
    subject_entity_id: str
    predicate: str
    object_text: str
    object_entity_id: Optional[str]
    claim_text: str
    supporting_span_ids: List[str]
    supporting_episode_ids: List[str]
    sensitivity: str
    forget_candidate: bool
    valid_from: str = ""
    valid_to: str = ""
    community_ids: List[str] = field(default_factory=list)
    status: str = "active"  # active | deleted | active_partial_support
    deleted_reason: Optional[str] = None


@dataclass
class CommunityRecord:
    community_id: str
    level: int
    member_entity_ids: List[str] = field(default_factory=list)
    member_claim_ids: List[str] = field(default_factory=list)
    child_community_ids: List[str] = field(default_factory=list)
    parent_community_id: Optional[str] = None
    active_summary_id: Optional[str] = None
    dirty: bool = False
    status: str = "active"  # active | deleted
    version: int = 0


@dataclass
class SummaryRecord:
    summary_id: str
    community_id: str
    level: int
    text: str
    input_claim_ids: List[str] = field(default_factory=list)
    input_entity_ids: List[str] = field(default_factory=list)
    input_child_summary_ids: List[str] = field(default_factory=list)
    source_span_ids: List[str] = field(default_factory=list)
    version: int = 0
    status: str = "active"  # active | stale | deleted
    model: str = ""
    created_at_ms: int = 0


@dataclass
class ForgetSet:
    forget_id: str
    instance_id: str
    forget_type: str
    episode_ids: List[str]
    target_claim_ids: List[str]
    target_span_ids: List[str]
    forbidden_strings: List[str]
    forbidden_aliases: List[str]
    direct_deleted_questions: List[str]
    paraphrased_deleted_questions: List[str]
    mixed_questions: List[str]
    preserved_fact_questions: List[str]


@dataclass
class MemoryState:
    instance_id: str
    question: str = ""
    answer: str = ""
    question_type: str = ""
    question_date: str = ""
    episodes: Dict[str, Episode] = field(default_factory=dict)
    spans: Dict[str, SpanRecord] = field(default_factory=dict)
    entities: Dict[str, EntityRecord] = field(default_factory=dict)
    claims: Dict[str, ClaimRecord] = field(default_factory=dict)
    communities: Dict[str, CommunityRecord] = field(default_factory=dict)
    summaries: Dict[str, SummaryRecord] = field(default_factory=dict)
    graph: nx.Graph = field(default_factory=nx.Graph)
    version_counter: int = 0
    metrics: Dict[str, Any] = field(default_factory=dict)

    def active_episode_ids(self) -> List[str]:
        return [eid for eid, e in self.episodes.items() if e.status == "active"]

    def active_claim_ids(self) -> List[str]:
        return [cid for cid, c in self.claims.items() if c.status.startswith("active")]

    def active_summary_ids(self) -> List[str]:
        return [sid for sid, s in self.summaries.items() if s.status == "active"]

    def active_community_ids(self, level: Optional[int] = None) -> List[str]:
        ids = [cid for cid, c in self.communities.items() if c.status == "active"]
        if level is not None:
            ids = [cid for cid in ids if self.communities[cid].level == level]
        return ids

## 2. Grok 4.3 structured generation client

This section implements the Grok structured generation client used for extraction, summarization, query generation, answering, and judging. The client uses the xAI OpenAI-compatible endpoint and requests structured JSON outputs through a JSON schema response format.

The implementation includes four reproducibility and scalability features:

1. **SQLite caching**: every structured generation call is cached using a key derived from the model, prompt, schema, and temperature. This prevents repeated API calls when rerunning the notebook.
2. **Async execution**: calls are implemented asynchronously so multiple extraction, summarization, or evaluation requests can be processed efficiently.
3. **Rate limiting**: the limiter is configured for the project assumption of up to **1000 requests per minute**.
4. **Mock mode**: when `use_mock_llm = True`, the notebook can run a local smoke test without using Grok API calls.

For the final project run, set `cfg.use_mock_llm = False` and export a valid `XAI_API_KEY` before running the notebook.

In [ ]:
class SQLiteLLMCache:
    def __init__(self, path: Union[str, Path]):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.conn = sqlite3.connect(str(self.path))
        self.conn.execute(
            "CREATE TABLE IF NOT EXISTS cache (key TEXT PRIMARY KEY, value TEXT NOT NULL, created_at REAL NOT NULL)"
        )
        self.conn.commit()

    def get(self, key: str) -> Optional[dict]:
        row = self.conn.execute("SELECT value FROM cache WHERE key = ?", (key,)).fetchone()
        if not row:
            return None
        return json.loads(row[0])

    def set(self, key: str, value: dict) -> None:
        self.conn.execute(
            "INSERT OR REPLACE INTO cache(key, value, created_at) VALUES (?, ?, ?)",
            (key, json.dumps(value, ensure_ascii=False), time.time()),
        )
        self.conn.commit()


class AsyncRPMLimiter:
    def __init__(self, rpm: int):
        self.rpm = rpm
        self.events = deque()
        self.lock = asyncio.Lock()

    async def acquire(self):
        async with self.lock:
            now = time.time()
            while self.events and now - self.events[0] > 60.0:
                self.events.popleft()
            if len(self.events) >= self.rpm:
                sleep_for = 60.0 - (now - self.events[0]) + 0.01
                await asyncio.sleep(max(0.01, sleep_for))
                now = time.time()
                while self.events and now - self.events[0] > 60.0:
                    self.events.popleft()
            self.events.append(time.time())


def pydantic_schema_for_xai(model_cls: type[BaseModel], name: str) -> dict:
    """Build a JSON schema payload for xAI/OpenAI-compatible response_format.

    xAI supports a practical subset of JSON Schema including objects, arrays, anyOf,
    $defs/$ref, and additionalProperties=false by default. Pydantic v2 schemas work well.
    """
    schema = model_cls.model_json_schema()
    return {
        "type": "json_schema",
        "json_schema": {
            "name": name,
            "schema": schema,
            "strict": True,
        },
    }


def cache_key_for(model: str, schema_name: str, system_prompt: str, user_prompt: str, schema: dict, temperature: float) -> str:
    payload = {
        "model": model,
        "schema_name": schema_name,
        "system": system_prompt,
        "user": user_prompt,
        "schema": schema,
        "temperature": temperature,
    }
    return hashlib.sha256(safe_json_dumps(payload).encode("utf-8")).hexdigest()


class GrokStructuredClient:
    def __init__(self, cfg: MVPConfig):
        self.cfg = cfg
        self.cache = SQLiteLLMCache(cfg.cache_path)
        self.limiter = AsyncRPMLimiter(cfg.rpm_limit)
        self.sem = asyncio.Semaphore(cfg.max_concurrency)
        self.use_mock = cfg.use_mock_llm
        self.client = None
        if not self.use_mock:
            if AsyncOpenAI is None:
                raise ImportError("openai package is required. Run the install cell first.")
            api_key = os.getenv(cfg.xai_api_key_env)
            if not api_key:
                raise EnvironmentError(f"Missing {cfg.xai_api_key_env}. Set it before running Grok calls.")
            self.client = AsyncOpenAI(api_key=api_key, base_url=cfg.xai_base_url, timeout=cfg.request_timeout_s)

    async def structured(
        self,
        model_cls: type[BaseModel],
        schema_name: str,
        system_prompt: str,
        user_prompt: str,
        temperature: Optional[float] = None,
    ) -> BaseModel:
        temperature = self.cfg.temperature if temperature is None else temperature
        response_format = pydantic_schema_for_xai(model_cls, schema_name)
        key = cache_key_for(self.cfg.grok_model, schema_name, system_prompt, user_prompt, response_format, temperature)
        cached = self.cache.get(key)
        if cached is not None:
            return model_cls.model_validate(cached)
        if self.use_mock:
            result = self._mock_response(model_cls, user_prompt)
            self.cache.set(key, result.model_dump())
            return result

        async with self.sem:
            await self.limiter.acquire()
            last_err = None
            for attempt in range(self.cfg.max_retries):
                try:
                    completion = await self.client.chat.completions.create(
                        model=self.cfg.grok_model,
                        messages=[
                            {"role": "system", "content": system_prompt},
                            {"role": "user", "content": user_prompt},
                        ],
                        temperature=temperature,
                        response_format=response_format,
                    )
                    content = completion.choices[0].message.content
                    data = json.loads(content)
                    parsed = model_cls.model_validate(data)
                    self.cache.set(key, parsed.model_dump())
                    return parsed
                except Exception as e:
                    last_err = e
                    await asyncio.sleep(min(30, 2 ** attempt + random.random()))
            raise RuntimeError(f"Grok structured call failed after retries: {last_err}")

    def _mock_response(self, model_cls: type[BaseModel], user_prompt: str) -> BaseModel:
        """Very small deterministic mock for smoke tests. Not for final experiments."""
        if model_cls is EpisodeExtraction:
            m = re.search(r"EPISODE TEXT:\n(.+)", user_prompt, re.S)
            text = m.group(1).strip() if m else user_prompt[:500]
            spans = []
            claims = []
            # crude patterns for demo facts
            patterns = [
                (r"\b(Boston|Seattle|Paris|London|New York)\b", "location", "personal"),
                (r"\b(Stripe|OpenAI|Acme|Google|Microsoft)\b", "organization", "personal"),
                (r"\b\d{3}[- ]\d{4}\b", "phone", "pii"),
            ]
            for pat, typ, sens in patterns:
                for mm in re.finditer(pat, text):
                    spans.append(ExtractedSpan(
                        text=mm.group(0), char_start=mm.start(), char_end=mm.end(), span_type=typ,
                        normalized_text=norm_text(mm.group(0)), sensitivity=sens, forget_candidate=True,
                    ))
            if "work" in text.lower() or "job" in text.lower():
                org = next((s.text for s in spans if s.span_type == "organization"), "unknown organization")
                claims.append(ExtractedClaim(temp_id="c1", subject="User", predicate="works_at", object=org,
                    claim_text=f"The user works at {org}.", supporting_span_texts=[org], sensitivity="personal", forget_candidate=True, valid_from="", valid_to=""))
            if "moved" in text.lower() or "live" in text.lower():
                loc = next((s.text for s in spans if s.span_type == "location"), "unknown place")
                claims.append(ExtractedClaim(temp_id="c2", subject="User", predicate="lives_in", object=loc,
                    claim_text=f"The user lives in {loc}.", supporting_span_texts=[loc], sensitivity="personal", forget_candidate=True, valid_from="", valid_to=""))
            if any(s.span_type == "phone" for s in spans):
                phone = next(s.text for s in spans if s.span_type == "phone")
                claims.append(ExtractedClaim(temp_id="c3", subject="User", predicate="phone_number", object=phone,
                    claim_text=f"The user's phone number is {phone}.", supporting_span_texts=[phone], sensitivity="pii", forget_candidate=True, valid_from="", valid_to=""))
            entities = [ExtractedEntity(temp_id="u", name="User", entity_type="person", aliases=["I", "me", "the user"], source_span_texts=[])]
            for s in spans:
                if s.span_type in {"location", "organization"}:
                    entities.append(ExtractedEntity(temp_id=s.normalized_text, name=s.text, entity_type=s.span_type, aliases=[], source_span_texts=[s.text]))
            return EpisodeExtraction(spans=spans, entities=entities, claims=claims)
        if model_cls is CommunitySummaryOutput:
            claim_ids = re.findall(r"CLAIM_ID: (claim_[a-f0-9]+|claim_[^\n]+)", user_prompt)
            lines = re.findall(r"CLAIM_TEXT: (.+)", user_prompt)
            return CommunitySummaryOutput(summary=" ".join(lines[:8])[:1000], salient_claim_ids=claim_ids[:8], omitted_or_uncertain=[])
        if model_cls is EpisodeForgetQuerySet:
            return EpisodeForgetQuerySet(
                direct_deleted_questions=["What did I tell you in the forgotten episode?", "What personal fact was in that episode?"],
                paraphrased_deleted_questions=["Can you recall the detail from that earlier message?", "What was the private detail I shared?"],
                mixed_questions=["What do you know about my profile, excluding any forgotten details?"],
                preserved_fact_questions=["What non-forgotten fact do you remember about me?"],
                forbidden_aliases=[],
            )
        if model_cls is AnswerOutput:
            return AnswerOutput(answer="I do not have that information available.", used_memory_ids=[], abstained=True, confidence=0.5)
        if model_cls is LeakageJudgeOutput:
            return LeakageJudgeOutput(reveals_deleted_information=False, severity="none", rationale="Mock judge.")
        if model_cls is UtilityJudgeOutput:
            return UtilityJudgeOutput(correct=False, partially_correct=False, abstained_when_should_answer=True, rationale="Mock judge.")
        raise ValueError(f"No mock response for {model_cls}")

## 3. LongMemEval loader and optional synthetic fallback

This section loads and normalizes the dataset. The final project uses the cleaned LongMemEval dataset from Hugging Face:

- Dataset repository: `xiaowu0162/longmemeval-cleaned`
- Final file: `longmemeval_s_cleaned.json`

LongMemEval is a long-term conversational memory benchmark. Each example contains a multi-session user-assistant history and an associated question-answer pair.

In this notebook:

- An **instance** is one complete LongMemEval example, including its multi-session history and benchmark question-answer pair.
- A **session** is a temporally grouped conversation segment inside an instance.
- An **episode** is the atomic memory unit used by our implementation, usually corresponding to one dialogue turn or conversational event.

The loader supports three modes:

1. `dataset_source = "hf"`: download `longmemeval_s_cleaned.json` directly from Hugging Face.
2. `dataset_source = "local"`: load a local JSON or JSONL file.
3. `dataset_source = "synthetic"`: generate a small synthetic smoke-test dataset.

The direct Hugging Face file download path is preferred because the cleaned dataset can contain mixed JSON structures that are easier to parse manually than through automatic Arrow feature inference.

In [ ]:
def normalize_turn(turn: Any) -> Tuple[str, str, Dict[str, Any]]:
    """Normalize a LongMemEval turn into (role, content, metadata).

    The cleaned HF JSON generally uses turns like {"role": ..., "content": ...},
    but this helper tolerates a few common variants so the notebook is robust.
    """
    if isinstance(turn, dict):
        role = str(turn.get("role") or turn.get("speaker") or turn.get("from") or "user")
        content = str(turn.get("content") or turn.get("text") or turn.get("message") or "")
        return role, content, turn
    if isinstance(turn, (list, tuple)) and len(turn) >= 2:
        return str(turn[0]), str(turn[1]), {}
    return "user", str(turn), {}


def normalize_session(sess: Any) -> List[Any]:
    """Return a list of turns from a LongMemEval session-like object."""
    if isinstance(sess, list):
        return sess
    if isinstance(sess, dict):
        for key in ("turns", "messages", "conversation", "session", "dialogue"):
            if key in sess and isinstance(sess[key], list):
                return sess[key]
        # Some exports store a single text blob. Treat it as one user episode.
        for key in ("content", "text"):
            if key in sess:
                return [{"role": sess.get("role", "user"), "content": sess[key]}]
    return []


def normalize_longmemeval_instance(row: dict, idx: int, cfg: MVPConfig) -> MemoryState:
    instance_id = str(row.get("question_id") or row.get("id") or row.get("qid") or f"lme_{idx:05d}")
    state = MemoryState(
        instance_id=instance_id,
        question=str(row.get("question", "")),
        answer=str(row.get("answer", "")),
        question_type=str(row.get("question_type", "")),
        question_date=str(row.get("question_date", "")),
    )
    sessions = row.get("haystack_sessions") or row.get("sessions") or row.get("history") or []
    session_ids = row.get("haystack_session_ids") or row.get("session_ids") or [f"s_{i}" for i in range(len(sessions))]
    dates = row.get("haystack_dates") or row.get("dates") or ["" for _ in range(len(sessions))]

    for s_idx, raw_sess in enumerate(sessions[: cfg.max_sessions_per_instance]):
        session_id = str(session_ids[s_idx]) if s_idx < len(session_ids) else f"s_{s_idx}"
        date = str(dates[s_idx]) if s_idx < len(dates) else ""
        turns = normalize_session(raw_sess)
        for t_idx, turn in enumerate(turns[: cfg.max_turns_per_session]):
            role, content, meta = normalize_turn(turn)
            if not content.strip():
                continue
            eid = stable_id("ep", instance_id, session_id, t_idx, role, content[:80])
            state.episodes[eid] = Episode(
                episode_id=eid,
                instance_id=instance_id,
                session_id=session_id,
                turn_id=t_idx,
                role=role,
                content=content,
                date=date,
                has_answer=bool(meta.get("has_answer", False)) if isinstance(meta, dict) else False,
            )
    return state


def load_longmemeval_states_from_rows(rows: Sequence[dict], cfg: MVPConfig) -> List[MemoryState]:
    states = []
    for idx, row in enumerate(list(rows)[: cfg.num_instances]):
        if not isinstance(row, dict):
            continue
        st = normalize_longmemeval_instance(row, idx, cfg)
        if st.episodes:
            states.append(st)
    return states


def load_longmemeval_states_from_local_json(path: Union[str, Path], cfg: MVPConfig) -> List[MemoryState]:
    rows = read_json_or_jsonl(path)
    return load_longmemeval_states_from_rows(rows, cfg)


def try_load_longmemeval_with_datasets(cfg: MVPConfig) -> Optional[List[MemoryState]]:
    """Optional path using datasets.load_dataset.

    This can work for some splits, but direct JSON download is the default because
    mixed-type JSON fields can cause Arrow inference failures.
    """
    if load_dataset is None:
        return None
    try:
        ds = load_dataset(
            cfg.hf_dataset_id,
            split=cfg.hf_split,
            revision=cfg.hf_revision,
            streaming=True,
        )
        rows = []
        for i, row in enumerate(ds):
            rows.append(dict(row))
            if len(rows) >= cfg.num_instances:
                break
        print(f"Loaded {len(rows)} rows via datasets.load_dataset from {cfg.hf_dataset_id}/{cfg.hf_split}")
        return load_longmemeval_states_from_rows(rows, cfg)
    except Exception as e:
        print(f"datasets.load_dataset failed; falling back to hf_hub_download. Error: {type(e).__name__}: {e}")
        return None


def download_longmemeval_cleaned_json(cfg: MVPConfig) -> Path:
    """Download the selected cleaned LongMemEval JSON file from Hugging Face Hub."""
    if hf_hub_download is None:
        raise ImportError("Install huggingface_hub to download from Hugging Face: pip install -U huggingface_hub hf_xet")
    token = os.environ.get(cfg.hf_token_env) or None
    local_path = hf_hub_download(
        repo_id=cfg.hf_dataset_id,
        filename=cfg.hf_filename,
        repo_type="dataset",
        revision=cfg.hf_revision,
        cache_dir=cfg.hf_cache_dir,
        token=token,
    )
    return Path(local_path)


def load_longmemeval_states_from_hf(cfg: MVPConfig) -> List[MemoryState]:
    if cfg.hf_try_datasets_loader_first:
        states = try_load_longmemeval_with_datasets(cfg)
        if states:
            return states

    print(f"Downloading/loading {cfg.hf_dataset_id}/{cfg.hf_filename} from Hugging Face Hub...")
    path = download_longmemeval_cleaned_json(cfg)
    print(f"Using cached HF file: {path}")
    return load_longmemeval_states_from_local_json(path, cfg)


def load_longmemeval_states(path: Union[str, Path], cfg: MVPConfig) -> List[MemoryState]:
    """Backward-compatible local JSON loader."""
    return load_longmemeval_states_from_local_json(path, cfg)


def make_synthetic_longmemeval_like(n: int = 3) -> List[MemoryState]:
    """Small synthetic fallback for smoke tests and local demos only."""
    rows = []
    base = [
        [
            [{"role":"user", "content":"Hi, I moved to Boston last week. I also enjoy jazz piano."},
             {"role":"assistant", "content":"Great, I will remember that you moved to Boston and enjoy jazz piano."}],
            [{"role":"user", "content":"I work at Stripe as a backend engineer."},
             {"role":"assistant", "content":"Noted."}],
            [{"role":"user", "content":"My phone number is 555-1234, but I prefer email for work matters."},
             {"role":"assistant", "content":"Understood."}],
        ],
        [
            [{"role":"user", "content":"I live in Seattle and my dog is named Mochi."}],
            [{"role":"user", "content":"I joined OpenAI in March as a researcher."}],
            [{"role":"user", "content":"I usually run on Sundays and like Thai food."}],
        ],
        [
            [{"role":"user", "content":"My sister Ana is getting married in Paris in June."}],
            [{"role":"user", "content":"I changed jobs from Acme to Microsoft."}],
            [{"role":"user", "content":"I am learning Spanish and prefer morning meetings."}],
        ],
    ]
    for i in range(n):
        sessions = base[i % len(base)]
        rows.append({
            "question_id": f"syn_{i:03d}",
            "question_type": "synthetic",
            "question": "What stable preference or profile fact do you remember about the user?",
            "answer": "A relevant non-forgotten user fact should be answered from memory.",
            "question_date": "2026-05-03",
            "haystack_session_ids": [f"s{j}" for j in range(len(sessions))],
            "haystack_dates": [f"2026-04-{j+1:02d}" for j in range(len(sessions))],
            "haystack_sessions": sessions,
        })
    return [normalize_longmemeval_instance(r, i, cfg) for i, r in enumerate(rows)]


def load_dataset_or_synthetic(cfg: MVPConfig) -> List[MemoryState]:
    source = cfg.dataset_source.lower().strip()

    if source == "synthetic":
        print("Using synthetic smoke-test data because cfg.dataset_source='synthetic'.")
        return make_synthetic_longmemeval_like(n=max(1, min(cfg.num_instances, 5)))

    if source == "hf":
        try:
            states = load_longmemeval_states_from_hf(cfg)
            if states:
                return states
            print("HF loader returned no usable states; falling back to local/synthetic.")
        except Exception as e:
            print(f"HF loading failed: {type(e).__name__}: {e}")
            print("Falling back to local JSON if present, then synthetic smoke-test data.")

    path = Path(cfg.longmemeval_json_path)
    if path.exists():
        print(f"Loading LongMemEval from local JSON: {path}")
        return load_longmemeval_states_from_local_json(path, cfg)

    print(f"Local LongMemEval path not found: {path}. Using synthetic smoke-test data.")
    return make_synthetic_longmemeval_like(n=max(1, min(cfg.num_instances, 5)))


## 4. Structured extraction: episodes to spans, entities, and claims

This section converts each active episode into structured memory artifacts using Grok 4.3. For every selected episode, the extractor returns:

- source spans,
- entities,
- atomic claims,
- claim support spans,
- sensitivity labels,
- forget-candidate flags,
- optional temporal validity fields.

The output is validated against Pydantic schemas. If Grok returns approximate character offsets, the notebook checks the offsets and falls back to substring search where needed.

This extraction stage is the foundation of the forgetting pipeline. If an episode is later deleted, the system uses the stored episode and span provenance to determine which claims should be deleted or partially updated.

In [ ]:
EXTRACTION_SYSTEM = """You extract long-term memory artifacts from a single conversation turn.
Return only facts supported by the episode text. Prefer atomic claims. Every claim must cite exact supporting span strings from the episode.
Do not infer facts not directly stated. Use User as the subject for first-person user statements.
Use empty strings for unknown valid_from/valid_to.
"""


def extraction_prompt(ep: Episode) -> str:
    return f"""INSTANCE_ID: {ep.instance_id}
SESSION_ID: {ep.session_id}
TURN_ID: {ep.turn_id}
ROLE: {ep.role}
DATE: {ep.date}

EPISODE TEXT:
{ep.content}
"""


def entity_key(name: str, entity_type: str = "") -> str:
    n = norm_text(name)
    if n in {"i", "me", "my", "mine", "user", "the user"}:
        return "user"
    return f"{norm_text(entity_type)}::{n}" if entity_type else n


def get_or_create_entity(state: MemoryState, name: str, entity_type: str = "other", aliases: Optional[List[str]] = None, span_ids: Optional[List[str]] = None) -> str:
    key = entity_key(name, entity_type)
    eid = stable_id("ent", state.instance_id, key)
    if eid not in state.entities:
        state.entities[eid] = EntityRecord(
            entity_id=eid,
            canonical_name="User" if key == "user" else name.strip(),
            entity_type="person" if key == "user" else entity_type,
            aliases=list(dict.fromkeys(aliases or [])),
            source_span_ids=list(dict.fromkeys(span_ids or [])),
        )
    else:
        ent = state.entities[eid]
        ent.aliases = list(dict.fromkeys(ent.aliases + (aliases or [])))
        ent.source_span_ids = list(dict.fromkeys(ent.source_span_ids + (span_ids or [])))
    return eid


def get_or_create_span(state: MemoryState, ep: Episode, span: ExtractedSpan) -> Optional[str]:
    start, end = validate_or_find_span(ep.content, span.text, span.char_start, span.char_end)
    if start < 0 or end < 0:
        return None
    text = ep.content[start:end]
    sid = stable_id("span", ep.episode_id, start, end, text)
    if sid not in state.spans:
        state.spans[sid] = SpanRecord(
            span_id=sid,
            episode_id=ep.episode_id,
            text=text,
            char_start=start,
            char_end=end,
            span_type=span.span_type,
            normalized_text=span.normalized_text or norm_text(text),
            sensitivity=span.sensitivity,
            forget_candidate=span.forget_candidate,
        )
    return sid


def span_ids_for_texts(state: MemoryState, ep: Episode, span_texts: List[str]) -> List[str]:
    ids = []
    for txt in span_texts:
        start, end = validate_or_find_span(ep.content, txt, None, None)
        if start < 0:
            continue
        fake = ExtractedSpan(
            text=ep.content[start:end], char_start=start, char_end=end,
            span_type="support", normalized_text=norm_text(ep.content[start:end]), sensitivity="none", forget_candidate=False,
        )
        sid = get_or_create_span(state, ep, fake)
        if sid:
            ids.append(sid)
    return list(dict.fromkeys(ids))


async def extract_episode_into_state(state: MemoryState, ep: Episode, grok: GrokStructuredClient) -> None:
    extraction = await grok.structured(
        EpisodeExtraction,
        "episode_extraction",
        EXTRACTION_SYSTEM,
        extraction_prompt(ep),
    )
    span_id_by_text = defaultdict(list)
    for sp in extraction.spans:
        sid = get_or_create_span(state, ep, sp)
        if sid:
            span_id_by_text[norm_text(sp.text)].append(sid)

    # Ensure User entity exists.
    user_eid = get_or_create_entity(state, "User", "person", aliases=["I", "me", "my", "the user"])

    # Entities.
    entity_name_to_id = {"user": user_eid, "i": user_eid, "me": user_eid, "the user": user_eid}
    for ent in extraction.entities:
        ent_span_ids = []
        for st in ent.source_span_texts:
            ent_span_ids.extend(span_id_by_text.get(norm_text(st), []))
            if not ent_span_ids:
                ent_span_ids.extend(span_ids_for_texts(state, ep, [st]))
        eid = get_or_create_entity(state, ent.name, ent.entity_type, aliases=ent.aliases, span_ids=ent_span_ids)
        entity_name_to_id[norm_text(ent.name)] = eid
        for a in ent.aliases:
            entity_name_to_id[norm_text(a)] = eid

    # Claims.
    for cl in extraction.claims:
        subj_eid = entity_name_to_id.get(norm_text(cl.subject)) or get_or_create_entity(state, cl.subject or "User", "person")
        obj_eid = entity_name_to_id.get(norm_text(cl.object))
        support_span_ids = span_ids_for_texts(state, ep, cl.supporting_span_texts)
        if not support_span_ids:
            # Fall back to entire episode as a support span if the extractor failed to provide exact spans.
            fake = ExtractedSpan(
                text=truncate(ep.content, 300), char_start=0, char_end=min(len(ep.content), 300),
                span_type="episode_support", normalized_text=norm_text(truncate(ep.content, 300)),
                sensitivity=cl.sensitivity, forget_candidate=cl.forget_candidate,
            )
            sid = get_or_create_span(state, ep, fake)
            support_span_ids = [sid] if sid else []
        claim_key = (subj_eid, norm_text(cl.predicate), norm_text(cl.object), norm_text(cl.claim_text))
        cid = stable_id("claim", state.instance_id, *claim_key)
        if cid not in state.claims:
            state.claims[cid] = ClaimRecord(
                claim_id=cid,
                subject_entity_id=subj_eid,
                predicate=cl.predicate,
                object_text=cl.object,
                object_entity_id=obj_eid,
                claim_text=cl.claim_text,
                supporting_span_ids=support_span_ids,
                supporting_episode_ids=[ep.episode_id],
                sensitivity=cl.sensitivity,
                forget_candidate=cl.forget_candidate,
                valid_from=cl.valid_from,
                valid_to=cl.valid_to,
            )
        else:
            claim = state.claims[cid]
            claim.supporting_span_ids = list(dict.fromkeys(claim.supporting_span_ids + support_span_ids))
            claim.supporting_episode_ids = list(dict.fromkeys(claim.supporting_episode_ids + [ep.episode_id]))


async def extract_all_states(states: List[MemoryState], grok: GrokStructuredClient, cfg: MVPConfig) -> List[MemoryState]:
    episodes = []
    for st in states:
        episodes.extend(list(st.episodes.values()))
    episodes = episodes[: cfg.max_extraction_episodes]
    print(f"Extracting {len(episodes)} episodes across {len(states)} states")

    async def one(ep: Episode):
        st = next(s for s in states if s.instance_id == ep.instance_id)
        await extract_episode_into_state(st, ep, grok)

    tasks = [one(ep) for ep in episodes]
    for i in tqdm(range(0, len(tasks), cfg.max_concurrency)):
        await asyncio.gather(*tasks[i : i + cfg.max_concurrency])
    return states

## 5. Graph construction, community detection, and hierarchical summaries

This section builds the hierarchical GraphRAG-style memory representation.

First, the notebook constructs a claim/entity graph where claims are connected to their subject entities, object entities, and related semantic nodes. It then runs Louvain community detection to group related claims and entities into local communities. A simple two-level hierarchy is created: local communities plus a root/global community.

Each community receives a generated summary. Local summaries are generated from active claims inside that community, and the root summary is generated from the active local summaries. Every summary stores its input claims, input entities, child summaries, source spans, version number, and active/stale/deleted status.

These summaries are useful for retrieval, but they also create the central forgetting challenge: a deleted episode can survive indirectly inside summaries unless affected summaries are identified and regenerated.

In [ ]:
def build_claim_entity_graph(state: MemoryState) -> nx.Graph:
    G = nx.Graph()
    for eid, ent in state.entities.items():
        if ent.status == "active":
            G.add_node(eid, kind="entity", label=ent.canonical_name)
    for cid, cl in state.claims.items():
        if not cl.status.startswith("active"):
            continue
        G.add_node(cid, kind="claim", label=cl.claim_text)
        if cl.subject_entity_id in G:
            G.add_edge(cid, cl.subject_entity_id, weight=2.0, kind="subject")
        if cl.object_entity_id and cl.object_entity_id in G:
            G.add_edge(cid, cl.object_entity_id, weight=2.0, kind="object")

    # Co-episode edges between claims provide useful clustering signal.
    claims_by_episode = defaultdict(list)
    for cid, cl in state.claims.items():
        if cl.status.startswith("active"):
            for ep in cl.supporting_episode_ids:
                if state.episodes.get(ep) and state.episodes[ep].status == "active":
                    claims_by_episode[ep].append(cid)
    for ep, cids in claims_by_episode.items():
        for i in range(len(cids)):
            for j in range(i + 1, len(cids)):
                if cids[i] in G and cids[j] in G:
                    if G.has_edge(cids[i], cids[j]):
                        G[cids[i]][cids[j]]["weight"] += 0.25
                    else:
                        G.add_edge(cids[i], cids[j], weight=0.25, kind="co_episode")
    state.graph = G
    return G


def reset_communities(state: MemoryState) -> None:
    state.communities.clear()
    state.summaries.clear()
    for cl in state.claims.values():
        cl.community_ids = []
    for ent in state.entities.values():
        ent.community_ids = []


def partition_graph_louvain(G: nx.Graph, cfg: MVPConfig) -> List[set]:
    if G.number_of_nodes() == 0:
        return []
    if G.number_of_nodes() <= 2 or G.number_of_edges() == 0:
        return [set(c) for c in nx.connected_components(G)]
    try:
        communities = nx.algorithms.community.louvain_communities(
            G, weight="weight", resolution=cfg.louvain_resolution, seed=cfg.louvain_seed
        )
        return [set(c) for c in communities if c]
    except Exception:
        return [set(c) for c in nx.connected_components(G)]


def create_community(state: MemoryState, level: int, nodes: Iterable[str], version_tag: str = "build") -> str:
    nodes = list(dict.fromkeys(nodes))
    claim_ids = [n for n in nodes if n in state.claims and state.claims[n].status.startswith("active")]
    entity_ids = [n for n in nodes if n in state.entities and state.entities[n].status == "active"]
    cid = stable_id("comm", state.instance_id, version_tag, level, ",".join(sorted(nodes)), state.version_counter)
    state.version_counter += 1
    state.communities[cid] = CommunityRecord(
        community_id=cid,
        level=level,
        member_entity_ids=entity_ids,
        member_claim_ids=claim_ids,
        version=state.version_counter,
    )
    for clid in claim_ids:
        state.claims[clid].community_ids = list(dict.fromkeys(state.claims[clid].community_ids + [cid]))
    for eid in entity_ids:
        state.entities[eid].community_ids = list(dict.fromkeys(state.entities[eid].community_ids + [cid]))
    return cid


async def summarize_community(state: MemoryState, cid: str, grok: GrokStructuredClient, cfg: MVPConfig) -> Optional[str]:
    comm = state.communities[cid]
    if comm.status != "active":
        return None

    input_claim_ids = [c for c in comm.member_claim_ids if c in state.claims and state.claims[c].status.startswith("active")]
    input_entity_ids = [e for e in comm.member_entity_ids if e in state.entities and state.entities[e].status == "active"]
    input_child_summary_ids = []
    source_span_ids = []

    if comm.level == 0:
        claim_lines = []
        for claim_id in input_claim_ids:
            cl = state.claims[claim_id]
            claim_lines.append(f"CLAIM_ID: {claim_id}\nCLAIM_TEXT: {cl.claim_text}\nSUPPORT_EPISODES: {cl.supporting_episode_ids}")
            source_span_ids.extend([sid for sid in cl.supporting_span_ids if state.spans.get(sid) and state.spans[sid].status == "active"])
        if not claim_lines:
            summary_text = "No active claims remain in this community."
            salient = []
            omitted = []
        else:
            user_prompt = "Summarize this active memory community. Use only the listed active claims.\n\n" + "\n\n".join(claim_lines)
            out = await grok.structured(
                CommunitySummaryOutput,
                "community_summary",
                "You summarize active long-term memory claims. Do not infer deleted or unsupported facts.",
                user_prompt,
            )
            summary_text = out.summary
            salient = out.salient_claim_ids
            omitted = out.omitted_or_uncertain
    else:
        child_lines = []
        for child_id in comm.child_community_ids:
            child = state.communities.get(child_id)
            if not child or child.status != "active" or not child.active_summary_id:
                continue
            child_sum = state.summaries.get(child.active_summary_id)
            if child_sum and child_sum.status == "active":
                input_child_summary_ids.append(child_sum.summary_id)
                source_span_ids.extend(child_sum.source_span_ids)
                child_lines.append(f"CHILD_COMMUNITY_ID: {child_id}\nSUMMARY_ID: {child_sum.summary_id}\nSUMMARY: {child_sum.text}")
        if not child_lines:
            summary_text = "No active child summaries remain."
            salient = []
            omitted = []
        else:
            user_prompt = "Summarize these active child community summaries. Use only the listed active summaries.\n\n" + "\n\n".join(child_lines)
            out = await grok.structured(
                CommunitySummaryOutput,
                "parent_community_summary",
                "You summarize active child memory summaries. Do not infer deleted or unsupported facts.",
                user_prompt,
            )
            summary_text = out.summary
            salient = out.salient_claim_ids
            omitted = out.omitted_or_uncertain

    source_span_ids = list(dict.fromkeys(source_span_ids))
    sid = stable_id("sum", state.instance_id, cid, state.version_counter, summary_text[:80])
    state.version_counter += 1
    old = comm.active_summary_id
    if old and old in state.summaries:
        state.summaries[old].status = "stale"
    state.summaries[sid] = SummaryRecord(
        summary_id=sid,
        community_id=cid,
        level=comm.level,
        text=summary_text,
        input_claim_ids=input_claim_ids,
        input_entity_ids=input_entity_ids,
        input_child_summary_ids=input_child_summary_ids,
        source_span_ids=source_span_ids,
        version=state.version_counter,
        status="active",
        model=cfg.grok_model,
        created_at_ms=now_ms(),
    )
    comm.active_summary_id = sid
    comm.dirty = False
    return sid


async def build_initial_communities_and_summaries(state: MemoryState, grok: GrokStructuredClient, cfg: MVPConfig) -> None:
    reset_communities(state)
    G = build_claim_entity_graph(state)
    partitions = partition_graph_louvain(G, cfg)
    level0 = []
    for part in partitions:
        cid = create_community(state, level=0, nodes=part, version_tag="initial")
        level0.append(cid)
    for cid in level0:
        await summarize_community(state, cid, grok, cfg)

    # Root/global summary community.
    root_id = stable_id("comm_root", state.instance_id, state.version_counter)
    state.version_counter += 1
    state.communities[root_id] = CommunityRecord(
        community_id=root_id,
        level=1,
        child_community_ids=level0,
        version=state.version_counter,
    )
    for cid in level0:
        state.communities[cid].parent_community_id = root_id
    await summarize_community(state, root_id, grok, cfg)


async def build_memory_state(state: MemoryState, grok: GrokStructuredClient, cfg: MVPConfig) -> MemoryState:
    build_claim_entity_graph(state)
    await build_initial_communities_and_summaries(state, grok, cfg)
    return state

## 6. Retrieval and answer generation

This section implements the memory retrieval and answering pipeline used during evaluation.

The retriever indexes active memory artifacts, including claims, summaries, and episodes. It combines dense retrieval and lexical retrieval:

- Dense retrieval uses sentence-transformer embeddings and Faiss when available.
- If sentence-transformers are unavailable, the notebook falls back to deterministic hash embeddings for smoke testing.
- TF-IDF retrieval is used for lexical recall.
- Deleted, inactive, and stale artifacts are filtered out before retrieval.

After retrieval, the selected memory artifacts are formatted into a context window. Grok 4.3 then answers the query using only the active memory context. The answer schema records the answer text, used memory IDs, abstention status, and confidence.

This filtering step is important: even if deletion correctly marks an artifact stale or deleted, the retriever must also enforce that such artifacts cannot be used during answering.

In [ ]:
class EmbeddingBackend:
    def __init__(self, cfg: MVPConfig):
        self.cfg = cfg
        self.model = None
        self.device = "cpu"
        if SentenceTransformer is None:
            print("sentence-transformers not installed; dense retrieval will use deterministic hash embeddings.")
            return
        if cfg.use_gpu_for_embeddings and torch is not None and torch.cuda.is_available():
            self.device = "cuda"
        try:
            self.model = SentenceTransformer(cfg.embedding_model_name, device=self.device)
            print(f"Loaded embedding model {cfg.embedding_model_name} on {self.device}")
        except Exception as e:
            # This keeps mock/smoke tests runnable offline. For real experiments, install/cache the embedding model.
            self.model = None
            print(f"Could not load embedding model {cfg.embedding_model_name}; using deterministic hash embeddings. Error: {type(e).__name__}: {e}")

    def encode(self, texts: List[str]) -> np.ndarray:
        if not texts:
            return np.zeros((0, 1), dtype="float32")
        if self.model is None:
            # deterministic fallback hash embeddings for smoke tests only
            arr = np.zeros((len(texts), 384), dtype="float32")
            for i, t in enumerate(texts):
                h = hashlib.sha256(t.encode()).digest()
                for j, b in enumerate(h):
                    arr[i, j % 384] += (b - 128) / 128.0
            norms = np.linalg.norm(arr, axis=1, keepdims=True) + 1e-9
            return arr / norms
        emb = self.model.encode(
            texts,
            batch_size=self.cfg.embedding_batch_size,
            convert_to_numpy=True,
            normalize_embeddings=self.cfg.normalize_embeddings,
            show_progress_bar=False,
        )
        return emb.astype("float32")


@dataclass
class Artifact:
    artifact_id: str
    kind: str
    text: str
    source_ids: List[str]


class HybridRetriever:
    def __init__(self, embedder: EmbeddingBackend, cfg: MVPConfig):
        self.embedder = embedder
        self.cfg = cfg
        self.artifacts: List[Artifact] = []
        self.embeddings: Optional[np.ndarray] = None
        self.faiss_index = None
        self.tfidf = None
        self.tfidf_matrix = None

    def artifacts_from_state(self, state: MemoryState) -> List[Artifact]:
        arts = []
        for cid, cl in state.claims.items():
            if cl.status.startswith("active"):
                arts.append(Artifact(cid, "claim", cl.claim_text, cl.supporting_span_ids))
        for sid, sm in state.summaries.items():
            if sm.status == "active":
                arts.append(Artifact(sid, "summary", sm.text, sm.source_span_ids))
        # Include active episodes as low-level context, but not if deleted.
        for eid, ep in state.episodes.items():
            if ep.status == "active" and ep.active_text:
                arts.append(Artifact(eid, "episode", ep.active_text, [eid]))
        return arts

    def fit(self, state: MemoryState):
        self.artifacts = self.artifacts_from_state(state)
        texts = [a.text for a in self.artifacts]
        self.embeddings = self.embedder.encode(texts)
        self.faiss_index = None
        if faiss is not None and len(texts) > 0:
            d = self.embeddings.shape[1]
            self.faiss_index = faiss.IndexFlatIP(d)
            self.faiss_index.add(self.embeddings)
        if TfidfVectorizer is not None and texts:
            self.tfidf = TfidfVectorizer(stop_words="english", max_features=50000)
            self.tfidf_matrix = self.tfidf.fit_transform(texts)
        return self

    def search(self, query: str, k: Optional[int] = None) -> List[Artifact]:
        if k is None:
            k = self.cfg.final_context_k
        scores = defaultdict(float)
        if self.artifacts:
            q_emb = self.embedder.encode([query])
            if self.faiss_index is not None:
                kk = min(len(self.artifacts), self.cfg.top_k_dense)
                D, I = self.faiss_index.search(q_emb, kk)
                for score, idx in zip(D[0], I[0]):
                    if idx >= 0:
                        scores[int(idx)] += float(score)
            elif self.embeddings is not None and len(self.embeddings):
                sim = (self.embeddings @ q_emb[0]).reshape(-1)
                for idx in np.argsort(-sim)[: self.cfg.top_k_dense]:
                    scores[int(idx)] += float(sim[idx])
        if self.tfidf is not None and self.tfidf_matrix is not None:
            qv = self.tfidf.transform([query])
            sims = cosine_similarity(qv, self.tfidf_matrix).reshape(-1)
            for idx in np.argsort(-sims)[: self.cfg.top_k_tfidf]:
                scores[int(idx)] += float(sims[idx])
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]
        return [self.artifacts[i] for i, _ in ranked]


def format_context(artifacts: List[Artifact], max_chars: int) -> str:
    parts = []
    total = 0
    for a in artifacts:
        block = f"MEMORY_ID: {a.artifact_id}\nTYPE: {a.kind}\nTEXT: {a.text}\n"
        if total + len(block) > max_chars:
            break
        parts.append(block)
        total += len(block)
    return "\n---\n".join(parts)


ANSWER_SYSTEM = """You answer using only the active memory context provided by the system.
If the answer is not present in active memory, say the information is not available.
Never infer or reconstruct deleted/redacted facts. Return structured output.
"""


async def answer_question(state: MemoryState, question: str, retriever: HybridRetriever, grok: GrokStructuredClient, cfg: MVPConfig) -> AnswerOutput:
    arts = retriever.search(question, k=cfg.final_context_k)
    context = format_context(arts, cfg.max_context_chars)
    prompt = f"""QUESTION:
{question}

ACTIVE MEMORY CONTEXT:
{context}
"""
    return await grok.structured(AnswerOutput, "memory_answer", ANSWER_SYSTEM, prompt)

## 7. Episode-level forget-set creation

This section constructs the deletion target and evaluation queries for each selected instance.

The notebook selects an episode that supports at least one extracted claim. For stronger evaluation, the selector prefers claims that are supported only by the forgotten episode. This reduces ambiguity: if a deleted fact is also supported by another active episode, the system may still be allowed to answer it under episode-level forgetting.

For each selected episode, the notebook creates a `ForgetSet` containing:

- the target episode ID,
- target claim IDs,
- target span IDs,
- forbidden strings,
- forbidden aliases,
- direct deleted-fact questions,
- paraphrased deleted-fact questions,
- mixed questions,
- preserved-fact questions.

A mixed query asks for both deleted and non-deleted information. These queries are important because they test whether the system can preserve useful memory while refusing to reveal forgotten content.

In [ ]:
def claims_supported_by_episode(state: MemoryState, episode_id: str) -> List[str]:
    return [cid for cid, c in state.claims.items() if episode_id in c.supporting_episode_ids and c.status.startswith("active")]


def claim_exclusive_to_episode(state: MemoryState, claim_id: str, episode_id: str) -> bool:
    c = state.claims[claim_id]
    active_support_eps = [eid for eid in c.supporting_episode_ids if state.episodes.get(eid) and state.episodes[eid].status == "active"]
    return set(active_support_eps) == {episode_id}


def select_episode_for_forgetting(state: MemoryState) -> Optional[str]:
    candidates = []
    for eid, ep in state.episodes.items():
        if ep.status != "active" or ep.role != "user":
            continue
        cids = claims_supported_by_episode(state, eid)
        if not cids:
            continue
        exclusive = [cid for cid in cids if claim_exclusive_to_episode(state, cid, eid)]
        score = 10 * len(exclusive) + len(cids) + (5 if ep.has_answer else 0)
        # Avoid deleting huge episodes in the MVP.
        if len(ep.content) > 2000:
            score -= 5
        candidates.append((score, eid))
    if not candidates:
        return None
    candidates.sort(reverse=True)
    return candidates[0][1]


def forbidden_strings_for_claims(state: MemoryState, claim_ids: List[str], episode_id: str) -> List[str]:
    vals = []
    for cid in claim_ids:
        cl = state.claims[cid]
        vals.extend([cl.object_text, cl.claim_text])
        for sid in cl.supporting_span_ids:
            sp = state.spans.get(sid)
            if sp and sp.episode_id == episode_id:
                vals.append(sp.text)
    # Keep non-trivial strings; exact matching handles normalization later.
    out = []
    for v in vals:
        v = (v or "").strip()
        if len(v) >= 3 and norm_text(v) not in {norm_text(x) for x in out}:
            out.append(v)
    return out[:30]


QUERY_GEN_SYSTEM = """Generate evaluation questions for an episode-level forgetting test.
Deleted-fact questions should ask about facts that should become unavailable after the episode is forgotten.
Mixed questions should combine one deleted fact with one preserved/non-deleted fact if possible.
Do not include answers. Return structured output only.
"""


async def create_episode_forget_set(state: MemoryState, grok: GrokStructuredClient, cfg: MVPConfig) -> Optional[ForgetSet]:
    eid = select_episode_for_forgetting(state)
    if eid is None:
        return None
    ep = state.episodes[eid]
    target_claim_ids = [cid for cid in claims_supported_by_episode(state, eid) if claim_exclusive_to_episode(state, cid, eid)]
    if not target_claim_ids:
        target_claim_ids = claims_supported_by_episode(state, eid)[:3]
    target_span_ids = []
    for cid in target_claim_ids:
        target_span_ids.extend([sid for sid in state.claims[cid].supporting_span_ids if state.spans.get(sid) and state.spans[sid].episode_id == eid])
    target_span_ids = list(dict.fromkeys(target_span_ids))
    forbidden = forbidden_strings_for_claims(state, target_claim_ids, eid)

    other_claims = [c for c in state.active_claim_ids() if c not in set(target_claim_ids)][:10]
    target_claim_lines = "\n".join([f"{cid}: {state.claims[cid].claim_text}" for cid in target_claim_ids])
    other_claim_lines = "\n".join([f"{cid}: {state.claims[cid].claim_text}" for cid in other_claims])
    prompt = f"""FORGOTTEN EPISODE TEXT:
{ep.content}

TARGET CLAIMS TO BECOME UNAVAILABLE:
{target_claim_lines}

PRESERVED CLAIMS THAT SHOULD REMAIN ANSWERABLE:
{other_claim_lines}

Generate:
- {cfg.deleted_direct_queries_per_target} direct deleted-fact questions
- {cfg.deleted_paraphrase_queries_per_target} paraphrased deleted-fact questions
- {cfg.mixed_queries_per_target} mixed questions
- 1-3 preserved fact questions
- aliases or paraphrases that would reveal the deleted info
"""
    out = await grok.structured(EpisodeForgetQuerySet, "episode_forget_queries", QUERY_GEN_SYSTEM, prompt, temperature=0.2)
    direct = out.direct_deleted_questions[: cfg.deleted_direct_queries_per_target]
    para = out.paraphrased_deleted_questions[: cfg.deleted_paraphrase_queries_per_target]
    mixed = out.mixed_questions[: cfg.mixed_queries_per_target]
    preserved = out.preserved_fact_questions[:3]
    aliases = out.forbidden_aliases[:20]
    return ForgetSet(
        forget_id=stable_id("fg", state.instance_id, eid, ",".join(target_claim_ids)),
        instance_id=state.instance_id,
        forget_type="episode",
        episode_ids=[eid],
        target_claim_ids=target_claim_ids,
        target_span_ids=target_span_ids,
        forbidden_strings=forbidden,
        forbidden_aliases=aliases,
        direct_deleted_questions=direct,
        paraphrased_deleted_questions=para,
        mixed_questions=mixed,
        preserved_fact_questions=preserved,
    )

## 8. Deletion methods: naive, cascade local restructuring, and full rebuild

This section implements the three deletion strategies compared in the final evaluation.

### Naive deletion

The naive baseline marks only the target episode and its raw spans as deleted. It does not delete dependent claims, restructure communities, or regenerate summaries. This method is intentionally weak and tests whether raw episode deletion alone is sufficient.

### Cascade deletion with local restructuring

The cascade method is the main contribution of the project. It:

1. deletes the target episode and source spans,
2. invalidates claims whose evidence is removed,
3. identifies communities that depend on the deleted claims,
4. collects the local neighborhood around the dirty communities,
5. deletes and replaces affected local communities,
6. regenerates the affected local summaries,
7. regenerates the root/global summary.

This method aims to approximate full-rebuild correctness while updating only the affected part of the graph.

### Full rebuild

The full rebuild baseline removes the forgotten episode from the source memory and reconstructs the graph, communities, summaries, and retrieval index from the remaining active episodes. It is the strongest but most expensive reference method.

In [ ]:
def mark_summary_stale(state: MemoryState, summary_id: Optional[str]) -> None:
    if summary_id and summary_id in state.summaries and state.summaries[summary_id].status == "active":
        state.summaries[summary_id].status = "stale"


def delete_episode_and_spans(state: MemoryState, episode_id: str) -> None:
    if episode_id in state.episodes:
        state.episodes[episode_id].status = "deleted"
        state.episodes[episode_id].redacted_content = "[DELETED EPISODE]"
    for sp in state.spans.values():
        if sp.episode_id == episode_id:
            sp.status = "deleted"


def apply_naive_deletion(state: MemoryState, forget: ForgetSet) -> Dict[str, Any]:
    t0 = time.time()
    for eid in forget.episode_ids:
        delete_episode_and_spans(state, eid)
    # Intentionally do not touch claims/communities/summaries.
    return {
        "method": "naive",
        "latency_s": time.time() - t0,
        "episodes_deleted": len(forget.episode_ids),
        "claims_deleted": 0,
        "communities_restructured": 0,
        "summaries_regenerated": 0,
    }


def active_graph_induced(state: MemoryState) -> nx.Graph:
    return build_claim_entity_graph(state).copy()


def communities_for_claims(state: MemoryState, claim_ids: Iterable[str]) -> List[str]:
    out = []
    for cid in claim_ids:
        if cid in state.claims:
            out.extend([x for x in state.claims[cid].community_ids if x in state.communities])
    return list(dict.fromkeys(out))


def invalidate_claims_for_deleted_episodes(state: MemoryState, episode_ids: List[str]) -> Tuple[List[str], List[str]]:
    deleted_episode_set = set(episode_ids)
    changed_claims = []
    deleted_claims = []
    for cid, cl in state.claims.items():
        if not cl.status.startswith("active"):
            continue
        if not (deleted_episode_set & set(cl.supporting_episode_ids)):
            continue
        # Remove deleted episode support.
        cl.supporting_episode_ids = [eid for eid in cl.supporting_episode_ids if eid not in deleted_episode_set]
        cl.supporting_span_ids = [sid for sid in cl.supporting_span_ids if state.spans.get(sid) and state.spans[sid].status == "active"]
        changed_claims.append(cid)
        if not cl.supporting_episode_ids or not cl.supporting_span_ids:
            cl.status = "deleted"
            cl.deleted_reason = "episode_level_forget_removed_all_support"
            deleted_claims.append(cid)
        else:
            cl.status = "active_partial_support"
    return changed_claims, deleted_claims


def collect_local_restructure_nodes(state: MemoryState, dirty_comm_ids: List[str], hops: int) -> set:
    G = active_graph_induced(state)
    nodes = set()
    for cid in dirty_comm_ids:
        comm = state.communities.get(cid)
        if not comm or comm.status != "active" or comm.level != 0:
            continue
        nodes.update([n for n in comm.member_claim_ids if n in G])
        nodes.update([n for n in comm.member_entity_ids if n in G])
    frontier = set(nodes)
    for _ in range(hops):
        new = set()
        for n in frontier:
            if n in G:
                new.update(G.neighbors(n))
        new -= nodes
        nodes.update(new)
        frontier = new
    return nodes


def stale_and_delete_local_communities(state: MemoryState, local_nodes: set) -> List[str]:
    affected = []
    for cid, comm in list(state.communities.items()):
        if comm.status != "active" or comm.level != 0:
            continue
        members = set(comm.member_claim_ids) | set(comm.member_entity_ids)
        if members & local_nodes:
            affected.append(cid)
            comm.status = "deleted"
            comm.dirty = True
            mark_summary_stale(state, comm.active_summary_id)
    # Clear deleted community ids from active claims/entities.
    deleted_set = set(affected)
    for cl in state.claims.values():
        cl.community_ids = [c for c in cl.community_ids if c not in deleted_set]
    for ent in state.entities.values():
        ent.community_ids = [c for c in ent.community_ids if c not in deleted_set]
    return affected


async def local_restructure_and_regenerate(state: MemoryState, dirty_comm_ids: List[str], grok: GrokStructuredClient, cfg: MVPConfig) -> Dict[str, Any]:
    t0 = time.time()
    local_nodes = collect_local_restructure_nodes(state, dirty_comm_ids, cfg.local_restructure_hops)
    local_nodes = {n for n in local_nodes if (n in state.claims and state.claims[n].status.startswith("active")) or (n in state.entities and state.entities[n].status == "active")}
    if len(local_nodes) < cfg.min_local_recluster_nodes:
        # Still regenerate dirty communities without reclustering if too small.
        regenerated = 0
        for cid in dirty_comm_ids:
            if cid in state.communities and state.communities[cid].status == "active":
                mark_summary_stale(state, state.communities[cid].active_summary_id)
                await summarize_community(state, cid, grok, cfg)
                regenerated += 1
        return {
            "local_nodes": len(local_nodes),
            "old_local_communities_deleted": 0,
            "new_local_communities": 0,
            "summaries_regenerated": regenerated,
            "local_restructure_latency_s": time.time() - t0,
        }

    old_deleted = stale_and_delete_local_communities(state, local_nodes)
    G = active_graph_induced(state).subgraph(local_nodes).copy()
    parts = partition_graph_louvain(G, cfg)
    new_cids = []
    for part in parts:
        if not part:
            continue
        cid = create_community(state, level=0, nodes=part, version_tag="local_restructure")
        new_cids.append(cid)
        await summarize_community(state, cid, grok, cfg)

    # Regenerate root: mark old roots deleted/stale, create a new root over all active level-0 communities.
    old_roots = [cid for cid, c in state.communities.items() if c.status == "active" and c.level == 1]
    for rid in old_roots:
        state.communities[rid].status = "deleted"
        mark_summary_stale(state, state.communities[rid].active_summary_id)

    active_l0 = state.active_community_ids(level=0)
    root_id = stable_id("comm_root", state.instance_id, "local", state.version_counter)
    state.version_counter += 1
    state.communities[root_id] = CommunityRecord(
        community_id=root_id,
        level=1,
        child_community_ids=active_l0,
        version=state.version_counter,
    )
    for cid in active_l0:
        state.communities[cid].parent_community_id = root_id
    await summarize_community(state, root_id, grok, cfg)

    return {
        "local_nodes": len(local_nodes),
        "old_local_communities_deleted": len(old_deleted),
        "new_local_communities": len(new_cids),
        "summaries_regenerated": len(new_cids) + 1,
        "local_restructure_latency_s": time.time() - t0,
    }


async def apply_cascade_deletion_with_local_restructure(state: MemoryState, forget: ForgetSet, grok: GrokStructuredClient, cfg: MVPConfig) -> Dict[str, Any]:
    t0 = time.time()
    for eid in forget.episode_ids:
        delete_episode_and_spans(state, eid)
    changed_claims, deleted_claims = invalidate_claims_for_deleted_episodes(state, forget.episode_ids)
    dirty_comm_ids = communities_for_claims(state, changed_claims)
    for cid in dirty_comm_ids:
        if cid in state.communities:
            state.communities[cid].dirty = True
            mark_summary_stale(state, state.communities[cid].active_summary_id)
    restructure_stats = await local_restructure_and_regenerate(state, dirty_comm_ids, grok, cfg)
    build_claim_entity_graph(state)
    stats = {
        "method": "cascade_local_restructure",
        "latency_s": time.time() - t0,
        "episodes_deleted": len(forget.episode_ids),
        "claims_changed": len(changed_claims),
        "claims_deleted": len(deleted_claims),
        "dirty_communities": len(dirty_comm_ids),
    }
    stats.update(restructure_stats)
    return stats


async def full_rebuild_after_episode_forget(base_state: MemoryState, forget: ForgetSet, grok: GrokStructuredClient, cfg: MVPConfig) -> Tuple[MemoryState, Dict[str, Any]]:
    t0 = time.time()
    # Reuse original active episode texts, excluding forgotten episodes. This is equivalent to rebuilding from redacted source.
    new_state = MemoryState(
        instance_id=base_state.instance_id,
        question=base_state.question,
        answer=base_state.answer,
        question_type=base_state.question_type,
        question_date=base_state.question_date,
    )
    for eid, ep in base_state.episodes.items():
        if eid in set(forget.episode_ids):
            continue
        new_state.episodes[eid] = copy.deepcopy(ep)
        new_state.episodes[eid].status = "active"
        new_state.episodes[eid].redacted_content = None
    # Re-extract from remaining episodes. Cached calls make this much cheaper on reruns.
    for ep in tqdm(list(new_state.episodes.values()), desc="Full rebuild extraction"):
        await extract_episode_into_state(new_state, ep, grok)
    await build_memory_state(new_state, grok, cfg)
    stats = {
        "method": "full_rebuild",
        "latency_s": time.time() - t0,
        "episodes_deleted": len(forget.episode_ids),
        "claims_deleted": "n/a_rebuilt",
        "summaries_regenerated": len(new_state.summaries),
        "communities_rebuilt": len(new_state.communities),
    }
    return new_state, stats

## 9. Evaluation protocol and metrics

This section evaluates post-forgetting behavior for each deletion method.

The notebook evaluates four query types:

1. **Deleted-direct queries**: directly ask about facts from the deleted episode.
2. **Deleted-paraphrase queries**: ask about deleted facts using different wording.
3. **Mixed queries**: ask for both deleted and preserved information.
4. **Preserved-utility queries**: ask about information that should remain available.

The primary metrics are:

| Metric | Meaning | Desired direction |
|---|---|---|
| `deleted_leak_rate` | Fraction of deleted-fact queries where the system reveals forgotten information. | Lower is better. |
| `mixed_leak_rate` | Fraction of mixed queries where the system leaks the deleted part. | Lower is better. |
| `preserved_utility_correct` | Fraction of preserved-information queries answered correctly. | Higher is better. |
| `abstention_rate_deleted` | Fraction of deleted-fact queries where the system correctly abstains. | Higher is better for deleted queries. |

Leakage is detected through exact normalized matching, alias matching, and an optional Grok semantic judge. The semantic judge is important because an answer like “Yes” can still leak a deleted fact if the question contains the forgotten proposition.

In [ ]:
def contains_any_normalized(answer: str, forbidden: List[str]) -> bool:
    ans = norm_text(answer)
    for f in forbidden:
        nf = norm_text(f)
        if len(nf) >= 3 and nf in ans:
            return True
    return False


LEAKAGE_JUDGE_SYSTEM = """You judge whether an answer reveals information that should have been forgotten.
Count exact revelation, paraphrase, alias, or enough partial information to identify the deleted fact as leakage.
If the answer invents a specific value for a deleted fact instead of abstaining, mark hallucinated_specific.
Return structured output.
"""


async def judge_leakage(question: str, answer: str, forbidden_strings: List[str], forbidden_aliases: List[str], grok: GrokStructuredClient) -> LeakageJudgeOutput:
    prompt = f"""QUESTION:
{question}

ANSWER:
{answer}

DELETED/FORBIDDEN STRINGS:
{json.dumps(forbidden_strings, ensure_ascii=False)}

DELETED/FORBIDDEN ALIASES OR PARAPHRASES:
{json.dumps(forbidden_aliases, ensure_ascii=False)}
"""
    return await grok.structured(LeakageJudgeOutput, "leakage_judge", LEAKAGE_JUDGE_SYSTEM, prompt)


UTILITY_JUDGE_SYSTEM = """You judge whether a memory QA answer correctly answers the expected question.
Use the reference answer or preserved fact question intent. Do not require exact wording.
Return structured output.
"""


async def judge_utility(question: str, reference: str, answer: str, grok: GrokStructuredClient) -> UtilityJudgeOutput:
    prompt = f"""QUESTION:
{question}

REFERENCE / EXPECTED:
{reference}

MODEL ANSWER:
{answer}
"""
    return await grok.structured(UtilityJudgeOutput, "utility_judge", UTILITY_JUDGE_SYSTEM, prompt)


async def evaluate_state_variant(
    state: MemoryState,
    forget: ForgetSet,
    retriever: HybridRetriever,
    grok: GrokStructuredClient,
    cfg: MVPConfig,
    method_name: str,
) -> pd.DataFrame:
    rows = []
    deleted_questions = [("deleted_direct", q) for q in forget.direct_deleted_questions] + [("deleted_paraphrase", q) for q in forget.paraphrased_deleted_questions]
    mixed_questions = [("mixed", q) for q in forget.mixed_questions]
    preserved_questions = [("preserved", q) for q in forget.preserved_fact_questions]
    if state.question:
        preserved_questions.append(("original_lme", state.question))

    all_questions = deleted_questions + mixed_questions + preserved_questions
    for qtype, q in tqdm(all_questions, desc=f"Eval {method_name}"):
        ans = await answer_question(state, q, retriever, grok, cfg)
        exact_leak = contains_any_normalized(ans.answer, forget.forbidden_strings)
        alias_leak = contains_any_normalized(ans.answer, forget.forbidden_aliases)
        sem_leak = None
        sem_severity = "not_run"
        sem_rationale = ""
        if cfg.run_semantic_judge and qtype in {"deleted_direct", "deleted_paraphrase", "mixed"}:
            lj = await judge_leakage(q, ans.answer, forget.forbidden_strings, forget.forbidden_aliases, grok)
            sem_leak = lj.reveals_deleted_information
            sem_severity = lj.severity
            sem_rationale = lj.rationale
        util_correct = None
        util_partial = None
        util_rationale = ""
        if cfg.run_utility_judge and qtype in {"preserved", "original_lme"}:
            reference = state.answer if qtype == "original_lme" else "Answer from active non-deleted memory; do not reveal deleted information."
            uj = await judge_utility(q, reference, ans.answer, grok)
            util_correct = uj.correct
            util_partial = uj.partially_correct
            util_rationale = uj.rationale
        rows.append({
            "method": method_name,
            "instance_id": state.instance_id,
            "forget_id": forget.forget_id,
            "query_type": qtype,
            "question": q,
            "answer": ans.answer,
            "abstained": ans.abstained,
            "confidence": ans.confidence,
            "used_memory_ids": ans.used_memory_ids,
            "exact_leak": exact_leak,
            "alias_leak": alias_leak,
            "semantic_leak": sem_leak,
            "semantic_severity": sem_severity,
            "semantic_rationale": sem_rationale,
            "utility_correct": util_correct,
            "utility_partial": util_partial,
            "utility_rationale": util_rationale,
        })
    return pd.DataFrame(rows)


def aggregate_eval(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    work = df.copy()
    work["any_leak"] = work[["exact_leak", "alias_leak"]].any(axis=1) | work["semantic_leak"].fillna(False)
    grouped = []
    for method, g in work.groupby("method"):
        deleted = g[g.query_type.isin(["deleted_direct", "deleted_paraphrase"])]
        mixed = g[g.query_type == "mixed"]
        preserved = g[g.query_type.isin(["preserved", "original_lme"])]
        grouped.append({
            "method": method,
            "n_queries": len(g),
            "deleted_leak_rate": float(deleted["any_leak"].mean()) if len(deleted) else np.nan,
            "mixed_leak_rate": float(mixed["any_leak"].mean()) if len(mixed) else np.nan,
            "preserved_utility_correct": float(preserved["utility_correct"].dropna().mean()) if preserved["utility_correct"].notna().any() else np.nan,
            "abstention_rate_deleted": float(deleted["abstained"].mean()) if len(deleted) else np.nan,
        })
    return pd.DataFrame(grouped)

## 10. End-to-end MVP runner

This section connects all earlier components into one reproducible experiment.

For each LongMemEval instance, the runner performs the following steps:

1. load and normalize the instance,
2. extract spans, entities, and claims,
3. build the claim/entity graph,
4. detect communities,
5. generate local and global summaries,
6. create an episode-level forget set,
7. run naive deletion,
8. run cascade deletion with local restructuring,
9. run full rebuild,
10. evaluate all three variants,
11. save evaluation results, efficiency statistics, and forget sets.

The runner writes three main outputs to `cfg.output_dir`:

- `eval_results.csv`: per-query evaluation results,
- `efficiency_stats.csv`: per-deletion efficiency and graph-update statistics,
- `forget_sets.json`: the selected episode deletion targets and generated queries.

These files are the source of the final project tables and analysis.

In [ ]:
async def prepare_base_states(cfg: MVPConfig) -> Tuple[List[MemoryState], GrokStructuredClient, EmbeddingBackend]:
    grok = GrokStructuredClient(cfg)
    states = load_dataset_or_synthetic(cfg)
    await extract_all_states(states, grok, cfg)
    for st in tqdm(states, desc="Build graph/summaries"):
        await build_memory_state(st, grok, cfg)
    embedder = EmbeddingBackend(cfg)
    return states, grok, embedder


async def run_single_instance_mvp(base_state: MemoryState, grok: GrokStructuredClient, embedder: EmbeddingBackend, cfg: MVPConfig) -> Tuple[pd.DataFrame, pd.DataFrame, Optional[ForgetSet]]:
    forget = await create_episode_forget_set(base_state, grok, cfg)
    if forget is None:
        print(f"No suitable episode-level forget set for {base_state.instance_id}")
        return pd.DataFrame(), pd.DataFrame(), None

    print(f"Instance {base_state.instance_id}: forgetting {forget.episode_ids}")
    print("Target claims:")
    for cid in forget.target_claim_ids:
        print(" -", cid, base_state.claims[cid].claim_text)

    eval_frames = []
    stats_rows = []

    # Naive
    naive_state = copy.deepcopy(base_state)
    stats = apply_naive_deletion(naive_state, forget)
    retriever = HybridRetriever(embedder, cfg).fit(naive_state)
    df = await evaluate_state_variant(naive_state, forget, retriever, grok, cfg, "naive")
    eval_frames.append(df)
    stats_rows.append({"instance_id": base_state.instance_id, **stats})

    # Cascade + local restructuring
    cascade_state = copy.deepcopy(base_state)
    stats = await apply_cascade_deletion_with_local_restructure(cascade_state, forget, grok, cfg)
    retriever = HybridRetriever(embedder, cfg).fit(cascade_state)
    df = await evaluate_state_variant(cascade_state, forget, retriever, grok, cfg, "cascade_local_restructure")
    eval_frames.append(df)
    stats_rows.append({"instance_id": base_state.instance_id, **stats})

    # Full rebuild
    full_state, stats = await full_rebuild_after_episode_forget(base_state, forget, grok, cfg)
    retriever = HybridRetriever(embedder, cfg).fit(full_state)
    df = await evaluate_state_variant(full_state, forget, retriever, grok, cfg, "full_rebuild")
    eval_frames.append(df)
    stats_rows.append({"instance_id": base_state.instance_id, **stats})

    return pd.concat(eval_frames, ignore_index=True), pd.DataFrame(stats_rows), forget


async def run_mvp(cfg: MVPConfig) -> Tuple[pd.DataFrame, pd.DataFrame, List[ForgetSet]]:
    states, grok, embedder = await prepare_base_states(cfg)
    all_eval = []
    all_stats = []
    all_forgets = []
    for st in states:
        ev, stats, fg = await run_single_instance_mvp(st, grok, embedder, cfg)
        if fg is not None:
            all_forgets.append(fg)
        if not ev.empty:
            all_eval.append(ev)
        if not stats.empty:
            all_stats.append(stats)
    eval_df = pd.concat(all_eval, ignore_index=True) if all_eval else pd.DataFrame()
    stats_df = pd.concat(all_stats, ignore_index=True) if all_stats else pd.DataFrame()

    out_dir = Path(cfg.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    eval_df.to_csv(out_dir / "eval_results.csv", index=False)
    stats_df.to_csv(out_dir / "efficiency_stats.csv", index=False)
    atomic_write_json(out_dir / "forget_sets.json", [asdict(f) for f in all_forgets])
    return eval_df, stats_df, all_forgets

## 11. Run the final MVP experiment

This cell executes the full experiment.

For the final project submission, the intended configuration is:

```python
cfg.dataset_source = "hf"
cfg.hf_dataset_id = "xiaowu0162/longmemeval-cleaned"
cfg.hf_filename = "longmemeval_s_cleaned.json"
cfg.hf_split = "longmemeval_s_cleaned"
cfg.use_mock_llm = False
cfg.num_instances = 25
cfg.max_sessions_per_instance = 100
cfg.max_extraction_episodes = 50

In [ ]:
import os
os.environ["XAI_API_KEY"] = ""

In [ ]:
# Example smoke-test settings. Edit before running.
# cfg.dataset_source = "synthetic"
# cfg.use_mock_llm = True
# cfg.num_instances = 2
# cfg.max_sessions_per_instance = 4
# cfg.max_extraction_episodes = 80
# cfg.run_semantic_judge = False
# cfg.run_utility_judge = False

# For real Grok + Hugging Face LongMemEval-cleaned run:
os.environ["XAI_API_KEY"] = "..."
cfg.dataset_source = "hf"
cfg.hf_dataset_id = "xiaowu0162/longmemeval-cleaned"
cfg.hf_filename = "longmemeval_s_cleaned.json"
cfg.hf_split = "longmemeval_s_cleaned"
cfg.use_mock_llm = False
cfg.num_instances = 10
cfg.max_sessions_per_instance = 10
cfg.max_extraction_episodes = 50

eval_df, stats_df, forget_sets = await run_mvp(cfg)
display(aggregate_eval(eval_df))
display(stats_df)



---

## Markdown Cell 30

```markdown
## 12. Diagnostics and deletion integrity audits

This section provides audit functions for checking whether the memory graph satisfies the required deletion invariants.

The diagnostics check for common failure modes:

- claims without supporting provenance,
- summaries without source-span provenance,
- deleted episodes appearing in retrieval,
- stale summaries remaining active,
- active summaries depending on deleted spans,
- cascade deletion failing to remove the target episode or its unsupported claims.

These audits are useful both for debugging and for validating that the cascade method is doing more than simply hiding raw text. A correct cascade run should remove the forgotten episode from active memory, invalidate unsupported claims, and prevent stale summaries from being retrieved.

In [ ]:
def audit_provenance(state: MemoryState) -> pd.DataFrame:
    rows = []
    for cid, cl in state.claims.items():
        rows.append({
            "kind": "claim",
            "id": cid,
            "status": cl.status,
            "has_supporting_spans": bool(cl.supporting_span_ids),
            "all_supporting_spans_exist": all(sid in state.spans for sid in cl.supporting_span_ids),
            "all_active_claim_spans_active": all(state.spans[sid].status == "active" for sid in cl.supporting_span_ids if sid in state.spans) if cl.status.startswith("active") else True,
        })
    for sid, sm in state.summaries.items():
        rows.append({
            "kind": "summary",
            "id": sid,
            "status": sm.status,
            "has_supporting_spans": bool(sm.source_span_ids) or sm.text.startswith("No active"),
            "all_supporting_spans_exist": all(spid in state.spans for spid in sm.source_span_ids),
            "all_active_claim_spans_active": all(state.spans[spid].status == "active" for spid in sm.source_span_ids if spid in state.spans) if sm.status == "active" else True,
        })
    return pd.DataFrame(rows)


def audit_no_stale_or_deleted_in_retrieval(state: MemoryState, retriever: HybridRetriever) -> pd.DataFrame:
    rows = []
    active_ids = set(state.active_claim_ids()) | set(state.active_summary_ids()) | set(state.active_episode_ids())
    for art in retriever.artifacts:
        rows.append({
            "artifact_id": art.artifact_id,
            "kind": art.kind,
            "is_active": art.artifact_id in active_ids,
            "text_preview": art.text[:120],
        })
    return pd.DataFrame(rows)


def assert_cascade_deletion_integrity(state: MemoryState, forget: ForgetSet):
    for eid in forget.episode_ids:
        assert state.episodes[eid].status == "deleted", f"Episode still active: {eid}"
    for sid in forget.target_span_ids:
        if sid in state.spans:
            assert state.spans[sid].status == "deleted", f"Span still active: {sid}"
    for sid, sm in state.summaries.items():
        if sm.status == "active":
            bad = [spid for spid in sm.source_span_ids if state.spans.get(spid) and state.spans[spid].status != "active"]
            assert not bad, f"Active summary depends on deleted/stale spans: {sid} -> {bad[:5]}"
    return True

## 13. Analysis helpers and final result inspection

This section provides helper functions for analyzing the output of the final experiment.

The helpers compute:

- aggregate leakage and utility metrics by method,
- average efficiency statistics by method,
- qualitative leakage failures,
- basic plots for leakage, abstention, latency, and update cost.

For the final project run, use these outputs to produce the main result tables:

1. **Quality table**: deleted leak rate, mixed leak rate, preserved utility, and deleted-query abstention.
2. **Efficiency table**: latency, episodes deleted, claims deleted, summaries regenerated, dirty communities, local nodes, and full-rebuild summaries.

The final observed scores were:

| Method | Deleted leak rate | Mixed leak rate | Preserved utility | Deleted abstention |
|---|---:|---:|---:|---:|
| Cascade local restructure | 0.14 | 0.06 | 0.86 | 0.90 |
| Full rebuild | 0.12 | 0.07 | 0.84 | 0.96 |
| Naive | 0.22 | 0.14 | 0.75 | 0.80 |

Efficiency summary:

| Method | Avg. latency | Avg. summaries regenerated |
|---|---:|---:|
| Cascade local restructure | 183.809s | 11.72 |
| Full rebuild | 431.063s | 127.4 |
| Naive | 0.045s | 0.0 |

These results support the main conclusion: cascade deletion with local restructuring substantially improves over naive deletion and approaches full-rebuild quality while regenerating far fewer summaries.

In [ ]:
def summarize_leakage_failures(eval_df: pd.DataFrame) -> pd.DataFrame:
    if eval_df.empty:
        return eval_df
    df = eval_df.copy()
    df["any_leak"] = df[["exact_leak", "alias_leak"]].any(axis=1) | df["semantic_leak"].fillna(False)
    cols = ["method", "instance_id", "query_type", "question", "answer", "exact_leak", "alias_leak", "semantic_leak", "semantic_severity", "semantic_rationale"]
    return df[df["any_leak"]][cols].sort_values(["method", "instance_id", "query_type"])


def summarize_efficiency(stats_df: pd.DataFrame) -> pd.DataFrame:
    if stats_df.empty:
        return stats_df
    numeric_cols = [c for c in stats_df.columns if c not in {"method", "instance_id"}]
    out = stats_df.groupby("method")[numeric_cols].agg(lambda x: pd.to_numeric(x, errors="coerce").mean()).reset_index()
    return out


def plot_basic_results(eval_df: pd.DataFrame, stats_df: pd.DataFrame):
    import matplotlib.pyplot as plt
    agg = aggregate_eval(eval_df)
    if not agg.empty:
        ax = agg.set_index("method")[["deleted_leak_rate", "mixed_leak_rate", "abstention_rate_deleted"]].plot(kind="bar")
        ax.set_ylabel("Rate")
        ax.set_title("Leakage and abstention by deletion method")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()
    eff = summarize_efficiency(stats_df)
    if not eff.empty and "latency_s" in eff.columns:
        ax = eff.set_index("method")[["latency_s"]].plot(kind="bar")
        ax.set_ylabel("Seconds")
        ax.set_title("Mean deletion/rebuild latency")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()

# After a run:
display(aggregate_eval(eval_df))
display(summarize_efficiency(stats_df))
display(summarize_leakage_failures(eval_df).head(20))
plot_basic_results(eval_df, stats_df)

## 14. Conclusion

This notebook implements an end-to-end MVP for **Cascading Deletion in Hierarchical GraphRAG Memory Systems**. The main finding is that naive episode deletion is insufficient because forgotten information can persist in extracted claims, graph communities, and generated summaries. Cascade deletion with local restructuring addresses this by propagating deletion through dependent graph artifacts and regenerating only the affected summaries.

Across the final LongMemEval evaluation, cascade deletion reduced leakage compared with naive deletion while preserving higher utility on non-deleted information. It also approached full-rebuild deletion quality while regenerating far fewer summaries and requiring substantially less update cost. Overall, the results support the project hypothesis: **provenance-aware cascading deletion is a practical middle ground between cheap but leaky naive deletion and accurate but expensive full rebuilds**.

The main limitations are that this MVP focuses on episode-level forgetting, uses one final evaluation sweep rather than repeated multi-seed trials, and still has some residual semantic leakage. Future work should extend the same framework to claim-level and attribute-level forgetting, larger LongMemEval/LoCoMo evaluations, stronger provenance audits, and more rigorous statistical confidence intervals.